# 📊 NeMo Fraud Detection: Chat Baseline Evaluation

This notebook performs a fair chat baseline evaluation of the unmodified base model (**Llama-3.1-8B-Instruct** via NVIDIA NIM) prior to Supervised Fine-Tuning (SFT).

### Features & W&B Lineage Integration:
1. **Upstream Artifact Lineage:** Dynamically consumes `curated-sft-dataset:latest` via `run.use_artifact(...)` to evaluate directly on the curated `validation.jsonl` split.
2. **Pre-Flight Check:** Verifies that the NIM endpoint is reachable (fail-fast).
3. **API Query:** Sends validated transcripts to the model via the chat completion endpoint.
4. **Label Mapping & Parsing:** Maps ground truth labels and robustly extracts model outputs using regex.
5. **Metrics, Tables & UI Output:** Computes accuracy, logs detailed W&B tables with `job_type="baseline-evaluation"`, and displays results in a formatted collapsible HTML block.

In [1]:
import json
import re
import requests
import wandb
import sys
import html
from pathlib import Path
from IPython.display import display, HTML

# Configuration of endpoints and default parameters
NIM_URL = "http://172.17.0.1:8800/v1/chat/completions"
MODEL_NAME = "meta/llama-3.1-8b-instruct"
INPUT_ARTIFACT_NAME = "curated-sft-dataset:latest"

print("✅ Libraries and configuration loaded successfully.")

✅ Libraries and configuration loaded successfully.


### 1. Pre-Flight Check (Connection Test to NIM Server)

In [2]:
def check_llm_connection():
    """Checks if the NIM service is reachable before starting the evaluation."""
    print("🔍 Checking connection to LLM (NIM service)...")
    try:
        health_url = "http://172.17.0.1:8800/v1/models"
        response = requests.get(health_url, timeout=5)
        if response.status_code == 200:
            print("✅ Connection to LLM established successfully!")
            return True
        else:
            print(f"❌ Connection failed. Status code: {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Unable to reach NIM service at '{NIM_URL}': {e}")
        return False

# Execute connection check immediately
if not check_llm_connection():
    raise SystemExit("Aborting: LLM server is not reachable.")

🔍 Checking connection to LLM (NIM service)...
✅ Connection to LLM established successfully!


### 2. Execution of Chat Baseline Evaluation & End-to-End W&B Lineage Tracking

In [3]:
def run_chat_baseline():
    # Initialize W&B run with explicit job_type for End-to-End Lineage
    run = wandb.init(
        project="nemo-fraud-detection", 
        job_type="baseline-evaluation",
        name="baseline-chat-evaluation",
        tags=["stage-3-baseline-eval", "nim", "llama-3.1-8b"],
        config={
            "model_name": MODEL_NAME,
            "input_artifact": INPUT_ARTIFACT_NAME,
            "temperature": 0.0,
            "max_tokens": 10
        }
    )
    
    # Consume upstream artifact from Curation step to establish lineage link
    print(f"📥 Consuming artifact '{INPUT_ARTIFACT_NAME}' from Weights & Biases...")
    artifact = run.use_artifact(INPUT_ARTIFACT_NAME)
    artifact_dir = artifact.download()
    val_file_path = Path(artifact_dir) / "validation.jsonl"
    
    print(f"🚀 Starting chat baseline evaluation on {val_file_path.name}...\n")
    
    correct = 0
    total = 0
    sample_log_lines = []
    
    # Create a W&B Table to log individual predictions and ground truth
    eval_table = wandb.Table(columns=["sample_index", "transcript", "true_label", "model_raw_output", "model_parsed_answer", "is_correct"])
    
    with open(val_file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            
            input_content = data.get("input", "")
            raw_true = data.get("output", "").strip().lower()
            
            # Standardize labels to English schema ('fraud' / 'legitimate')
            if "betrug" in raw_true and "kein" not in raw_true:
                true_label = "fraud"
            elif "fraud" in raw_true:
                true_label = "fraud"
            else:
                true_label = "legitimate"
            
            if not input_content:
                continue
                
            payload = {
                "model": MODEL_NAME,
                "messages": [
                    {
                        "role": "system",
                        "content": "You are a fraud detection assistant for customer support calls. Analyze the transcript and reply EXCLUSIVELY with exactly one word: 'fraud' or 'legitimate'."
                    },
                    {
                        "role": "user",
                        "content": input_content
                    }
                ],
                "temperature": 0.0,
                "max_tokens": 10
            }
            
            try:
                response = requests.post(NIM_URL, json=payload)
                result = response.json()
                
                if "choices" not in result:
                    print(f"API error at sample {total + 1}: {result}")
                    continue

                # Extract model response from chat structure
                model_raw = result["choices"][0]["message"]["content"].strip().lower()
                
                # Robust parsing via regex
                if "legitimate" in model_raw and "fraud" not in model_raw:
                    model_answer = "legitimate"
                elif "fraud" in model_raw and "legitimate" not in model_raw:
                    model_answer = "fraud"
                else:
                    match = re.search(r'\b(fraud|legitimate)\b', model_raw)
                    model_answer = match.group(1) if match else model_raw
                
                is_correct = (true_label == model_answer)
                if is_correct:
                    correct += 1
                total += 1
                
                # Add row to W&B Table
                eval_table.add_data(total, input_content[:200] + "...", true_label, model_raw, model_answer, is_correct)
                
                log_line = f"Sample {total:02d} | Expected: {true_label:<10} | Predicted: {model_answer:<10} | Correct: {is_correct}"
                sample_log_lines.append(log_line)
                print(log_line)
                
            except Exception as e:
                print(f"Error processing sample {total + 1}: {e}")

    accuracy = (correct / total) * 100 if total > 0 else 0
    
    # Log overall aggregate metrics and evaluation table to wandb
    run.log({
        "eval/accuracy": accuracy,
        "eval/correct_predictions": correct,
        "eval/total_samples": total,
        "eval/predictions_table": eval_table
    })
    run.finish()

    # Render output inside a collapsible HTML block labeled 'Sample Output'
    summary_text = (
        "=============================================\n"
        "📊 CHAT BASELINE EVALUATION RESULTS\n"
        "=============================================\n"
        f"✅ Correctly identified: {correct} / {total}\n"
        f"🎯 Accuracy:             {accuracy:.2f}%\n"
        "=============================================\n\n"
        "Detailed Samples:\n" +
        "\n".join(sample_log_lines)
    )
    
    escaped_output = html.escape(summary_text)
    html_widget = f"""
<details style="border: 1px solid #d0d7de; border-radius: 6px; padding: 10px 14px; background-color: #f6f8fa; margin-top: 10px;" open>
    <summary style="cursor: pointer; font-weight: bold; color: #0969da; font-size: 14px;">
        ▶ Sample Output
    </summary>
    <pre style="margin-top: 12px; white-space: pre-wrap; font-size: 13px; font-family: monospace; background: #ffffff; padding: 10px; border: 1px solid #e1e4e8; border-radius: 4px;">{escaped_output}</pre>
</details>
"""
    display(HTML(html_widget))

# Start evaluation
run_chat_baseline()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /root/.netrc.
wandb: Currently logged in as: lohmann-daniel (lohmann-daniel-myself) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


📥 Consuming artifact 'curated-sft-dataset:latest' from Weights & Biases...


wandb:   3 of 3 files downloaded.  


🚀 Starting chat baseline evaluation on validation.jsonl...

Sample 01 | Expected: legitimate | Predicted: fraud      | Correct: False
Sample 02 | Expected: fraud      | Predicted: fraud      | Correct: True
Sample 03 | Expected: fraud      | Predicted: fraud      | Correct: True
Sample 04 | Expected: legitimate | Predicted: legitimate | Correct: True
Sample 05 | Expected: fraud      | Predicted: fraud      | Correct: True
Sample 06 | Expected: fraud      | Predicted: fraud      | Correct: True
Sample 07 | Expected: fraud      | Predicted: legitimate | Correct: False
Sample 08 | Expected: legitimate | Predicted: legitimate | Correct: True
Sample 09 | Expected: fraud      | Predicted: fraud      | Correct: True
Sample 10 | Expected: legitimate | Predicted: legitimate | Correct: True
Sample 11 | Expected: fraud      | Predicted: legitimate | Correct: False
Sample 12 | Expected: fraud      | Predicted: fraud      | Correct: True


eval/accuracy,▁
eval/correct_predictions,▁
eval/total_samples,▁
eval/accuracy,75
eval/correct_predictions,9
eval/total_samples,12
